# BART + NLI Pipeline for Indonesian Abstractive Summarization

Notebook ini disiapkan untuk dijalankan di **Kaggle** (tanpa script terpisah).

Urutan kerja:
1. Cek GPU
2. Install dependency
3. Konversi dataset Liputan-6 ke JSONL
4. Preprocessing
5. Training BART
6. Generate summary + NLI reranking
7. Evaluasi ROUGE dan factual consistency

## 1. Cek GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Install Dependency

In [ ]:
!pip install -q transformers datasets evaluate rouge-score accelerate sentencepiece

## 3. Setup Directory

In [ ]:
from pathlib import Path
import os

WORK_DIR = Path("./results/thesis_pipeline")
DATA_DIR = WORK_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = WORK_DIR / "outputs" / "bart-baseline"

for d in [DATA_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Working dir:", WORK_DIR)

## 4. Konfigurasi Eksperimen

In [ ]:
MODEL_NAME        = "facebook/bart-base"
NLI_MODEL_NAME    = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
SUBSET            = "canonical"   # "canonical" atau "xtreme"
TEXT_COLUMN       = "article"     # kolom teks di dataset Liputan-6
SUMMARY_COLUMN    = "summary"

# Training hyperparameter
MAX_SOURCE_LENGTH = 512  # turun dari 768 agar lebih cepat
MAX_TARGET_LENGTH = 128
TRAIN_BATCH_SIZE  = 8   # naik dari 4
EVAL_BATCH_SIZE   = 8   # naik dari 4
LEARNING_RATE     = 2e-5
NUM_EPOCHS        = 3
GRAD_ACCUM_STEPS  = 2   # turun dari 4
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1
SEED              = 42

# Generate
NUM_CANDIDATES    = 4
ALPHA             = 0.7

BASE_DATASET = Path("./data/liputan6_raw")

print("Config OK")

## 5. Konversi Dataset Liputan-6 ke JSONL

Menggabungkan file JSON individual menjadi `train.jsonl`, `valid.jsonl`, `test.jsonl`.

In [ ]:
import json

SPLIT_MAP = {"train": "train", "dev": "valid", "test": "test"}

def tokens_to_text(sentences):
    return " ".join(" ".join(sent) for sent in sentences)

for src_split, dst_split in SPLIT_MAP.items():
    split_dir = BASE_DATASET / SUBSET / src_split
    output_file = DATA_DIR / f"{dst_split}.jsonl"

    if output_file.exists():
        print(f"{dst_split}.jsonl sudah ada, skip.")
        continue

    if not split_dir.exists():
        print(f"Folder tidak ditemukan: {split_dir}")
        continue

    json_files = sorted(split_dir.glob("*.json"))
    print(f"{src_split}: {len(json_files)} files → {output_file.name}")

    with output_file.open("w", encoding="utf-8") as out:
        for jf in json_files:
            with jf.open("r", encoding="utf-8") as f:
                data = json.load(f)
            row = {
                "id": data["id"],
                TEXT_COLUMN: tokens_to_text(data["clean_article"]),
                SUMMARY_COLUMN: tokens_to_text(data["clean_summary"]),
            }
            out.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\nKonversi selesai.")

## 6. Preprocessing (Filter & Normalisasi)

In [ ]:
import re
import unicodedata
from dataclasses import dataclass, field

WHITESPACE_RE = re.compile(r"\s+")
CONTROL_RE = re.compile(r"[\u0000-\u0008\u000B\u000C\u000E-\u001F\u007F]")

@dataclass
class PreprocessStats:
    total_rows: int = 0
    kept_rows: int = 0
    dropped_empty: int = 0
    dropped_too_short: int = 0
    dropped_duplicates: int = 0

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = CONTROL_RE.sub(" ", text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = WHITESPACE_RE.sub(" ", text)
    return text.strip()

def is_valid_pair(document, summary, min_document_chars=100, min_summary_chars=20):
    return bool(document) and bool(summary) and len(document) >= min_document_chars and len(summary) >= min_summary_chars

def preprocess_jsonl(input_file, output_file):
    stats = PreprocessStats()
    seen_pairs = set()
    cleaned_rows = []

    with Path(input_file).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            stats.total_rows += 1
            document = normalize_text(str(row.get(TEXT_COLUMN, "")))
            summary = normalize_text(str(row.get(SUMMARY_COLUMN, "")))

            if not document or not summary:
                stats.dropped_empty += 1
                continue
            if not is_valid_pair(document, summary):
                stats.dropped_too_short += 1
                continue
            pair_key = (document, summary)
            if pair_key in seen_pairs:
                stats.dropped_duplicates += 1
                continue
            seen_pairs.add(pair_key)
            cleaned_row = dict(row)
            cleaned_row[TEXT_COLUMN] = document
            cleaned_row[SUMMARY_COLUMN] = summary
            cleaned_rows.append(cleaned_row)
            stats.kept_rows += 1

    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_file).open("w", encoding="utf-8") as f:
        for row in cleaned_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"{Path(input_file).name} → kept {stats.kept_rows}/{stats.total_rows} rows")
    return stats

for split in ["train", "valid", "test"]:
    preprocess_jsonl(DATA_DIR / f"{split}.jsonl", PROCESSED_DIR / f"{split}.jsonl")

print("\nPreprocessing selesai.")

## 7. Training Baseline BART

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

def load_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

set_seed(SEED)

datasets = DatasetDict(
    train=load_jsonl(PROCESSED_DIR / "train.jsonl"),
    validation=load_jsonl(PROCESSED_DIR / "valid.jsonl"),
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess_fn(batch):
    model_inputs = tokenizer(
        batch[TEXT_COLUMN], max_length=MAX_SOURCE_LENGTH, truncation=True
    )
    labels = tokenizer(
        text_target=batch[SUMMARY_COLUMN], max_length=MAX_TARGET_LENGTH, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = datasets.map(
    preprocess_fn,
    batched=True,
    remove_columns=datasets["train"].column_names,
    desc="Tokenizing dataset",
)

total_steps = (len(tokenized["train"]) // (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_steps=warmup_steps,
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Training selesai. Model disimpan ke:", OUTPUT_DIR)

## 8. Generate Summary for BART Baseline (1 kandidat, tanpa NLI)

In [ ]:
import math
from transformers import AutoModelForSequenceClassification

class NLIScorer:
    def __init__(self, model_name, device=None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device).eval()
        id2label = {int(k): v.lower() for k, v in self.model.config.id2label.items()}
        self.entailment_idx = next(i for i, l in id2label.items() if "entail" in l)
        self.contradiction_idx = next(i for i, l in id2label.items() if "contrad" in l)
        self.neutral_idx = next(i for i, l in id2label.items() if "neutral" in l)

    @torch.inference_mode()
    def score(self, premise, hypothesis):
        enc = self.tokenizer(premise, hypothesis, truncation=True, max_length=512, return_tensors="pt")
        enc = {k: v.to(self.device) for k, v in enc.items()}
        probs = torch.softmax(self.model(**enc).logits[0], dim=-1)
        return {
            "entailment": float(probs[self.entailment_idx]),
            "contradiction": float(probs[self.contradiction_idx]),
            "neutral": float(probs[self.neutral_idx]),
        }


class BartSummarizer:
    def __init__(self, model_path, device=None):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device).eval()

    @torch.inference_mode()
    def generate_candidates(self, document, max_source_length=768, max_target_length=128,
                             min_target_length=32, num_beams=4, num_candidates=4):
        inputs = self.tokenizer(document, truncation=True, max_length=max_source_length, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        generated = self.model.generate(
            **inputs,
            max_length=max_target_length,
            min_length=min_target_length,
            num_beams=max(num_beams, num_candidates),
            num_return_sequences=num_candidates,
            length_penalty=1.0,
            early_stopping=True,
            output_scores=True,
            return_dict_in_generate=True,
        )
        texts = self.tokenizer.batch_decode(generated.sequences, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=True)
        return [{"summary": t.strip(), "generation_score": float(s)}
                for t, s in zip(texts, generated.sequences_scores.tolist())]

    def rerank_with_nli(self, document, candidates, nli_scorer, alpha=0.7):
        best = None
        for item in candidates:
            nli = nli_scorer.score(document, item["summary"])
            norm_gen = math.tanh(item["generation_score"] / 10.0)
            combined = alpha * nli["entailment"] + (1 - alpha) * norm_gen
            enriched = {**item, **nli, "combined_score": combined}
            if best is None or enriched["combined_score"] > best["combined_score"]:
                best = enriched
        return best


def generate_summaries(model_path, input_file, output_file, nli_model_name,
                        num_candidates=1, alpha=0.0):
    summarizer = BartSummarizer(model_path)
    nli_scorer = NLIScorer(nli_model_name)
    outputs = []

    with Path(input_file).open("r", encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]

    for i, row in enumerate(rows):
        document = row[TEXT_COLUMN]
        candidates = summarizer.generate_candidates(
            document, num_candidates=num_candidates
        )
        if num_candidates == 1:
            nli = nli_scorer.score(document, candidates[0]["summary"])
            best = {**candidates[0], **nli, "combined_score": 0.0}
        else:
            best = summarizer.rerank_with_nli(document, candidates, nli_scorer, alpha)
        outputs.append({
            "id": row.get("id"),
            "document": document,
            "reference_summary": row.get(SUMMARY_COLUMN),
            "generated_summary": best["summary"],
            "generation_score": best["generation_score"],
            "entailment_score": best.get("entailment", 0.0),
            "contradiction_score": best.get("contradiction", 0.0),
            "neutral_score": best.get("neutral", 0.0),
            "combined_score": best.get("combined_score", 0.0),
        })
        if (i + 1) % 100 == 0:
            print(f"Generated {i+1}/{len(rows)}", flush=True)

    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_file).open("w", encoding="utf-8") as f:
        for row in outputs:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Saved {len(outputs)} predictions → {output_file}", flush=True)


# Baseline: 1 kandidat, dengan NLI scoring (tanpa reranking)
generate_summaries(
    model_path=str(OUTPUT_DIR),
    input_file=PROCESSED_DIR / "test.jsonl",
    output_file=WORK_DIR / "outputs" / "predictions_baseline.jsonl",
    nli_model_name=NLI_MODEL_NAME,
    num_candidates=1,
    alpha=0.0,
)

## 9. Evaluasi BART Baseline

In [ ]:
import evaluate
import statistics

def evaluate_predictions(prediction_file, nli_model_name):
    rouge = evaluate.load("rouge")
    nli_scorer = NLIScorer(nli_model_name)

    with Path(prediction_file).open("r", encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]

    predictions = [r["generated_summary"] for r in rows]
    references  = [r["reference_summary"] for r in rows]
    rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=False)

    entailments, contradictions = [], []
    for i, row in enumerate(rows):
        result = nli_scorer.score(row["document"], row["generated_summary"])
        entailments.append(result["entailment"])
        contradictions.append(result["contradiction"])
        if (i + 1) % 100 == 0:
            print(f"NLI scored {i+1}/{len(rows)}")

    print("\nROUGE evaluation")
    print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")
    print("\nFactual consistency evaluation")
    print(f"Average entailment:    {statistics.mean(entailments):.4f}")
    print(f"Average contradiction: {statistics.mean(contradictions):.4f}")

evaluate_predictions(
    prediction_file=WORK_DIR / "outputs" / "predictions_baseline.jsonl",
    nli_model_name=NLI_MODEL_NAME,
)

## 10. Generate Summary + NLI Reranking

In [ ]:
generate_summaries(
    model_path=str(OUTPUT_DIR),
    input_file=PROCESSED_DIR / "test.jsonl",
    output_file=WORK_DIR / "outputs" / "predictions_nli.jsonl",
    nli_model_name=NLI_MODEL_NAME,
    num_candidates=NUM_CANDIDATES,
    alpha=ALPHA,
)

## 11. Evaluasi BART + NLI

In [ ]:
evaluate_predictions(
    prediction_file=WORK_DIR / "outputs" / "predictions_nli.jsonl",
    nli_model_name=NLI_MODEL_NAME,
)

## 12. Lihat Contoh Hasil Prediksi

In [ ]:
prediction_path = WORK_DIR / "outputs" / "predictions_nli.jsonl"
with prediction_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        row = json.loads(line)
        print("ID:", row.get("id"))
        print("Generated:", row.get("generated_summary"))
        print("Reference:", row.get("reference_summary"))
        print("Entailment:", row.get("entailment_score"))
        print("-" * 80)
        if i >= 2:
            break

## 13. Catatan Eksperimen

Untuk pelaporan jurnal, minimal bandingkan:
- BART baseline
- BART + NLI reranking

Catat juga:
- model summarization yang dipakai
- model NLI yang dipakai
- subset data (`canonical` atau `xtreme`)
- jumlah kandidat summary
- nilai `alpha`
- hasil ROUGE dan entailment

**Catatan Kaggle:**
- Download hasil dari tab **Output** setelah session selesai
- Kaggle memberi **30 jam GPU/minggu** — gunakan dengan efisien
- Aktifkan **"Save & Run All (Commit)"** agar output tersimpan permanen